# 07. Frequency-Domain Features & Correlation Analysis

## Objective

This notebook extracts frequency-domain wearable features from the preprocessed Parkinson's Disease Smartwatch Dataset (PADS) recordings and identifies strongly correlated features.

It follows the established preprocessing pipeline by reading the participant-level compressed `.npz` files produced after:

1. L1 trend filtering of accelerometer axes using `λ = 50`.
2. CLARABEL optimization.
3. Removal of the first 48 samples.
4. Re-zeroing of the time channel.
5. Preservation of gyroscope signals.
6. Calculation of accelerometer and gyroscope magnitudes.

The notebook produces:

- a frequency-domain feature table;
- separate accelerometer and gyroscope feature groups;
- a Pearson correlation matrix;
- a table of strongly correlated wearable features; and
- a summary of the correlation-analysis results.


## Frequency-Domain Features

For each accelerometer and gyroscope axis and magnitude signal, the following features are extracted:

- **Dominant frequency:** frequency with the highest spectral power.
- **Spectral centroid:** power-weighted average frequency.
- **Spectral entropy:** normalized measure of spectral complexity.
- **Spectral power:** total power represented by the one-sided power spectral density.

The analysis includes:

- Accelerometer X, Y, Z, and magnitude.
- Gyroscope X, Y, Z, and magnitude.

This yields 16 accelerometer and 16 gyroscope frequency-domain features per recording.


In [ ]:
# Libraries and project paths
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/ and src/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.frequency_features import (
    build_correlation_summary,
    build_frequency_feature_table,
    compute_feature_correlations,
    get_accelerometer_feature_columns,
    get_frequency_feature_columns,
    get_gyroscope_feature_columns,
    summarize_strong_correlations,
)

NPZ_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed_signals"
FEATURE_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "frequency_domain_features.csv"
)
CORRELATION_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "correlation_analysis"
)
CORRELATION_MATRIX_OUTPUT = (
    CORRELATION_OUTPUT_DIR / "frequency_feature_correlation_matrix.csv"
)
STRONG_CORRELATIONS_OUTPUT = (
    CORRELATION_OUTPUT_DIR / "strongly_correlated_features.csv"
)
CORRELATION_SUMMARY_OUTPUT = (
    CORRELATION_OUTPUT_DIR / "correlation_analysis_summary.csv"
)

STRONG_CORRELATION_THRESHOLD = 0.80

print("Project root:", PROJECT_ROOT)
print("Processed NPZ directory:", NPZ_DIR)
print("Frequency feature output:", FEATURE_OUTPUT)
print("Correlation output directory:", CORRELATION_OUTPUT_DIR)


## 1. Confirm Input Availability

The expected input is one participant-level file per participant in:

`data/processed/preprocessed_signals/`

Each file should follow the pattern `*_preprocessed.npz`.


In [ ]:
# Validate the processed-signal input directory
# =============================================================================

if not NPZ_DIR.exists():
    raise FileNotFoundError(
        f"Could not locate {NPZ_DIR}. "
        "Run or obtain the signal-preprocessing outputs before continuing."
    )

npz_files = sorted(NPZ_DIR.glob("*_preprocessed.npz"))

if not npz_files:
    raise FileNotFoundError(
        f"No participant .npz files were found in {NPZ_DIR}."
    )

print("Participant NPZ files found:", len(npz_files))
print("First file:", npz_files[0].name)


## 2. Inspect the Preprocessed `.npz` Structure

The structure is checked before dataset-wide extraction to confirm that the expected metadata, recording keys, and signal columns are available.


In [ ]:
# Inspect one participant file
# =============================================================================

with np.load(npz_files[0], allow_pickle=False) as sample:
    sample_columns = sample["__columns__"].astype(str).tolist()
    sample_recording_keys = (
        sample["__recording_keys__"].astype(str).tolist()
    )
    first_recording_key = sample_recording_keys[0]
    first_recording_shape = sample[first_recording_key].shape

print("Signal columns:", sample_columns)
print("Recordings in sample participant:", len(sample_recording_keys))
print("First recording key:", first_recording_key)
print("First recording shape:", first_recording_shape)

expected_columns = [
    "Time",
    "Accelerometer_X",
    "Accelerometer_Y",
    "Accelerometer_Z",
    "Gyroscope_X",
    "Gyroscope_Y",
    "Gyroscope_Z",
    "Acc_Magnitude",
    "Gyro_Magnitude",
]

missing_columns = [
    column for column in expected_columns if column not in sample_columns
]
print("Required columns missing:", missing_columns)
assert not missing_columns


## 3. Extract Dataset-Wide Frequency-Domain Features

One output row is produced for every participant × task × wrist recording. Sampling frequency is inferred separately from each recording's time channel using the median positive sampling interval.


In [ ]:
# Extract frequency-domain features
# =============================================================================

frequency_features = build_frequency_feature_table(NPZ_DIR)

print("Frequency feature table shape:", frequency_features.shape)
print("Participants:", frequency_features["patient_id"].nunique())
print("Tasks:", frequency_features["task"].nunique())
print("Wrists:", frequency_features["wrist"].nunique())

frequency_features.head()


In [ ]:
# Confirm expected frequency-feature groups
# =============================================================================

all_frequency_columns = get_frequency_feature_columns(frequency_features)
accelerometer_frequency_columns = (
    get_accelerometer_feature_columns(frequency_features)
)
gyroscope_frequency_columns = (
    get_gyroscope_feature_columns(frequency_features)
)

feature_group_summary = pd.DataFrame(
    {
        "feature_group": [
            "Accelerometer",
            "Gyroscope",
            "Total",
        ],
        "number_of_features": [
            len(accelerometer_frequency_columns),
            len(gyroscope_frequency_columns),
            len(all_frequency_columns),
        ],
    }
)

feature_group_summary


In [ ]:
# Display accelerometer and gyroscope feature groups
# =============================================================================

print("Accelerometer frequency features:")
for column in accelerometer_frequency_columns:
    print(" -", column)

print("\nGyroscope frequency features:")
for column in gyroscope_frequency_columns:
    print(" -", column)


## 4. Frequency-Feature Quality Assurance

The following checks confirm that identifiers are unique at the recording level and summarize missing or non-finite feature values.


In [ ]:
# Recording-level uniqueness and completeness checks
# =============================================================================

identifier_columns = ["patient_id", "task", "wrist"]
duplicate_recordings = frequency_features.duplicated(
    subset=identifier_columns,
    keep=False,
)

feature_qa = pd.DataFrame(
    {
        "metric": [
            "Rows",
            "Unique participant-task-wrist records",
            "Duplicate participant-task-wrist records",
            "Frequency features",
            "Missing frequency-feature values",
            "Infinite frequency-feature values",
        ],
        "value": [
            len(frequency_features),
            frequency_features[identifier_columns]
            .drop_duplicates()
            .shape[0],
            int(duplicate_recordings.sum()),
            len(all_frequency_columns),
            int(
                frequency_features[all_frequency_columns]
                .isna()
                .sum()
                .sum()
            ),
            int(
                np.isinf(
                    frequency_features[all_frequency_columns].to_numpy(
                        dtype=float
                    )
                ).sum()
            ),
        ],
    }
)

feature_qa


In [ ]:
# Feature-level missingness
# =============================================================================

feature_missingness = (
    frequency_features[all_frequency_columns]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

feature_missingness["missing_percentage"] = (
    feature_missingness["missing_values"]
    / len(frequency_features)
    * 100
).round(2)

feature_missingness.sort_values(
    "missing_values",
    ascending=False,
).head(20)


## 5. Save the Frequency-Domain Feature Table

The complete table is saved as:

`data/processed/frequency_domain_features.csv`


In [ ]:
# Save and verify frequency-domain feature table
# =============================================================================

FEATURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
frequency_features.to_csv(FEATURE_OUTPUT, index=False)

if not FEATURE_OUTPUT.exists() or FEATURE_OUTPUT.stat().st_size == 0:
    raise IOError(
        f"Frequency feature table was not saved correctly: {FEATURE_OUTPUT}"
    )

verified_frequency_features = pd.read_csv(
    FEATURE_OUTPUT,
    dtype={"patient_id": str},
)

assert verified_frequency_features.shape == frequency_features.shape

print("Saved:", FEATURE_OUTPUT)
print("Verified shape:", verified_frequency_features.shape)


## 6. Correlation Analysis

Pearson correlations are calculated across the 32 frequency-domain wearable features. Identifier and metadata variables are excluded.

A feature pair is classified as strongly correlated when:

`|r| ≥ 0.80`

The threshold can be changed through `STRONG_CORRELATION_THRESHOLD`.


In [ ]:
# Calculate the correlation matrix
# =============================================================================

correlation_matrix = compute_feature_correlations(
    frequency_features,
    method="pearson",
)

print("Correlation matrix shape:", correlation_matrix.shape)
correlation_matrix.round(3)


In [ ]:
# Visualize the frequency-feature correlation matrix
# =============================================================================

figure, axis = plt.subplots(figsize=(15, 13))
image = axis.imshow(
    correlation_matrix.to_numpy(),
    aspect="auto",
    vmin=-1,
    vmax=1,
)

axis.set_xticks(range(len(correlation_matrix.columns)))
axis.set_xticklabels(
    correlation_matrix.columns,
    rotation=90,
    fontsize=7,
)
axis.set_yticks(range(len(correlation_matrix.index)))
axis.set_yticklabels(
    correlation_matrix.index,
    fontsize=7,
)
axis.set_title("Frequency-Domain Wearable Feature Correlations")
figure.colorbar(image, ax=axis, label="Pearson correlation")
figure.tight_layout()
plt.show()


## 7. Identify Strongly Correlated Wearable Features

Duplicate and self-correlations are excluded. Each retained pair is classified as:

- accelerometer-only;
- gyroscope-only; or
- cross-sensor.


In [ ]:
# Identify strongly correlated feature pairs
# =============================================================================

strong_correlations = summarize_strong_correlations(
    correlation_matrix,
    threshold=STRONG_CORRELATION_THRESHOLD,
)

print(
    "Strongly correlated pairs:",
    len(strong_correlations),
)
strong_correlations.head(30)


In [ ]:
# Summarize strong correlations by relationship group and direction
# =============================================================================

if strong_correlations.empty:
    strong_correlation_group_summary = pd.DataFrame(
        columns=["relationship_group", "direction", "feature_pairs"]
    )
else:
    strong_correlation_group_summary = (
        strong_correlations
        .groupby(
            ["relationship_group", "direction"],
            as_index=False,
        )
        .size()
        .rename(columns={"size": "feature_pairs"})
    )

strong_correlation_group_summary


## 8. Save Correlation Deliverables

The following outputs are saved:

- `outputs/tables/correlation_analysis/frequency_feature_correlation_matrix.csv`
- `outputs/tables/correlation_analysis/strongly_correlated_features.csv`
- `outputs/tables/correlation_analysis/correlation_analysis_summary.csv`


In [ ]:
# Build and save the correlation-analysis summary
# =============================================================================

correlation_summary = build_correlation_summary(
    feature_table=frequency_features,
    strong_correlations=strong_correlations,
    threshold=STRONG_CORRELATION_THRESHOLD,
)

CORRELATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

correlation_matrix.to_csv(CORRELATION_MATRIX_OUTPUT)
strong_correlations.to_csv(
    STRONG_CORRELATIONS_OUTPUT,
    index=False,
)
correlation_summary.to_csv(
    CORRELATION_SUMMARY_OUTPUT,
    index=False,
)

output_paths = [
    CORRELATION_MATRIX_OUTPUT,
    STRONG_CORRELATIONS_OUTPUT,
    CORRELATION_SUMMARY_OUTPUT,
]

for output_path in output_paths:
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise IOError(f"Output was not saved correctly: {output_path}")
    print("Saved:", output_path)

correlation_summary


## 9. Final Summary

Upon successful execution, this notebook demonstrates that:

- frequency-domain features were extracted from the existing preprocessed signals;
- dominant frequency, spectral centroid, spectral entropy, and spectral power were calculated;
- accelerometer and gyroscope frequency-feature groups were created;
- the complete frequency-domain feature table was saved;
- the wearable-feature correlation matrix was produced;
- strongly correlated feature pairs were identified and grouped; and
- all requested correlation-analysis outputs were saved and verified.

Interpretation of the strongest correlations should consider whether highly related variables provide duplicate information. Strongly correlated features may be reviewed during later feature-selection and model-development stages rather than removed automatically at this step.
